# Class 2. Lexical Structure & Literal System

코드를 이루는 Token과 값의 표기법인 Literal을 다룬다. 식별자 명명 규칙을 익히고, `tokenize`로 코드를 나누어 본 뒤 PEP 8에 맞게 정리한다.


## 1. Token의 종류와 구분

### 1.1 Token 복습

Token은 코드를 이루는 가장 작은 의미 단위다. 문장을 단어로 나누듯이 `x = 10 + 5`를 `x`, `=`, `10`, `+`, `5`로 나눌 수 있다. 어휘 분석(Lexical Analysis)은 이런 Token을 만들고, 구문 분석(Syntax Analysis)은 Token 사이의 문법 관계를 다룬다.


### 1.2 Token의 4가지 종류

이 수업에서는 다음 네 범주로 묶어 살펴본다.

| 종류 | 역할 | 예 |
| --- | --- | --- |
| Keyword | 문법에서 용도가 정해진 단어 | `if`, `for`, `def`, `class` |
| Identifier | 변수·함수·클래스 등에 붙이는 이름 | `age`, `calculate`, `Person` |
| Literal | 코드에 직접 적는 값의 표기 | `42`, `"Hello"` |
| Operator / Delimiter | 연산과 코드 구조를 나타내는 기호 | `+`, `-`, `=`, `()`, `[]`, `{}` |


In [1]:
# Token 확인하기
import tokenize
import io

code = "x = 10 + 5"
tokens = tokenize.generate_tokens(io.StringIO(code).readline)

print("=== Token 분석 ===")
print(f"코드: {code}")
print("\nToken 목록:")

for token in tokens:
  if token.type not in [tokenize.ENCODING, tokenize.ENDMARKER]:
    print(f"  {tokenize.tok_name[token.type]:10} : '{token.string}'")

=== Token 분석 ===
코드: x = 10 + 5

Token 목록:
  NAME       : 'x'
  OP         : '='
  NUMBER     : '10'
  OP         : '+'
  NUMBER     : '5'
  NEWLINE    : ''


### 1.3 Keyword (예약어)

Keyword는 Python 문법에서 용도가 정해진 단어다. 변수명이나 함수명으로 쓸 수 없으며 대소문자를 구분한다. 예를 들어 `if`는 Keyword지만 `If`는 아니다. 현재 실행 중인 Python의 Keyword 목록은 아래 코드로 확인한다.


In [1]:
# Python의 모든 Keyword 확인
import keyword

print("=== Python Keywords ===")
print(f"총 개수: {len(keyword.kwlist)}개\n")

# 분류별로 출력
categories = {
    '제어 흐름': ['if', 'elif', 'else', 'for', 'while', 'break', 'continue', 'pass'],
    '함수/클래스': ['def', 'class', 'return', 'yield', 'lambda'],
    '논리 연산': ['and', 'or', 'not', 'is', 'in'],
    '예외 처리': ['try', 'except', 'finally', 'raise', 'assert'],
    '모듈': ['import', 'from', 'as'],
    '기타': ['del', 'global', 'nonlocal', 'with', 'async', 'await', 'True', 'False', 'None']
}

for category, keywords in categories.items():
  print(f"{category}:")
  for kw in keywords:
    print(f"  - {kw}")
  print()

=== Python Keywords ===
총 개수: 35개

제어 흐름:
  - if
  - elif
  - else
  - for
  - while
  - break
  - continue
  - pass

함수/클래스:
  - def
  - class
  - return
  - yield
  - lambda

논리 연산:
  - and
  - or
  - not
  - is
  - in

예외 처리:
  - try
  - except
  - finally
  - raise
  - assert

모듈:
  - import
  - from
  - as

기타:
  - del
  - global
  - nonlocal
  - with
  - async
  - await
  - True
  - False
  - None



#### Hard Keyword vs Soft Keyword

`if`, `for`, `def`처럼 항상 예약된 단어를 Hard Keyword라고 한다. 반면 Soft Keyword는 특정 문맥에서만 Keyword로 해석되며, 다른 문맥에서는 식별자로 쓸 수 있다. `match/case` 문의 `match`, `case`, `_`가 그 예다.


In [2]:
# Keyword 확인
print("=== Keyword 체크 ===")
print(f"'if'는 키워드? {keyword.iskeyword('if')}")
print(f"'If'는 키워드? {keyword.iskeyword('If')}")
print(f"'print'는 키워드? {keyword.iskeyword('print')}")
print()

# Soft Keyword
print("=== Soft Keyword ===")
print(f"'match'는 키워드? {keyword.iskeyword('match')}")
print(f"'match'는 Soft 키워드? {keyword.issoftkeyword('match')}")
print()

# Soft Keyword는 변수명으로 사용 가능
match = "문자열 매칭"  # 사용 가능
print(f"match 변수: {match}")
print()

# Hard Keyword는 불가능
# if = 10  # 오류: SyntaxError!
print("Hard Keyword는 변수명으로 사용 불가")

=== Keyword 체크 ===
'if'는 키워드? True
'If'는 키워드? False
'print'는 키워드? False

=== Soft Keyword ===
'match'는 키워드? False
'match'는 Soft 키워드? True

match 변수: 문자열 매칭

Hard Keyword는 변수명으로 사용 불가


### 1.4 Identifier (식별자)

Identifier는 프로그래머가 붙이는 이름이다. 변수의 `age`, `total_price`, 함수의 `calculate`, 클래스의 `Person`, 모듈의 `utils` 등이 해당한다.


#### 명명 규칙 (Rules - 필수)

**반드시 지켜야 하는 규칙**:

1. **사용 가능 문자**: 영문자(a-z, A-Z), 숫자(0-9), 언더스코어(\_)
2. **첫 글자**: 영문자 또는 \_ (숫자 불가)
3. **Keyword 금지**: 예약어 사용 불가
4. **대소문자 구분**: Name과 name은 다름
5. **특수문자 금지**: @, #, $, % 등 사용 불가


In [11]:
# 올바른 Identifier
print("=== 올바른 Identifier ===")

age = 25                  # 사용 가능
user_name = "Alice"       # 사용 가능
_temp = 100               # 사용 가능
count2 = 5                # 사용 가능
MyClass = type('MyClass', (), {})  # 사용 가능

print(f"age: {age}")
print(f"user_name: {user_name}")
print(f"_temp: {_temp}")
print(f"count2: {count2}")
print()

# 잘못된 Identifier (주석 처리)
print("=== 잘못된 Identifier ===")
# 2age = 10          # 오류: 숫자로 시작
# user-name = "x"    # 오류: 하이픈 사용
# for = 10           # 오류: 키워드 사용
# my var = 20        # 오류: 공백 포함
print("위 예시들은 모두 SyntaxError 발생")

=== 올바른 Identifier ===
age: 25
user_name: Alice
_temp: 100
count2: 5

=== 잘못된 Identifier ===
위 예시들은 모두 SyntaxError 발생


#### 명명 관례 (Convention - 권장)

**PEP 8 스타일 가이드**:

| 대상     | 관례       | 예시                          |
| -------- | ---------- | ----------------------------- |
| 변수명   | snake_case | user_age, total_count         |
| 함수명   | snake_case | calculate_total(), get_info() |
| 클래스명 | PascalCase | Person, BankAccount           |
| 상수     | UPPER_CASE | MAX_SIZE, PI                  |
| 모듈명   | snake_case | utils.py, database.py         |
| Private  | \_leading  | \_internal_var                |
| Magic    | **double** | \_\_init\_\_, \_\_str\_\_     |


In [ ]:
# PEP 8 명명 관례
print("=== PEP 8 명명 관례 ===")
print()

# 변수 - snake_case
user_age = 25
total_price = 1000
print(f"변수: user_age={user_age}, total_price={total_price}")

# 함수 - snake_case


def calculate_total(price, quantity):
  return price * quantity


print(f"함수: calculate_total(100, 5) = {calculate_total(100, 5)}")

# 클래스 - PascalCase


class UserAccount:
  def __init__(self, name):
    self.name = name


user = UserAccount("Bob")
print(f"클래스: UserAccount, 인스턴스 이름: {user.name}")

# 상수 - UPPER_CASE
MAX_SIZE = 1000
PI = 3.14159
print(f"상수: MAX_SIZE={MAX_SIZE}, PI={PI}")

### 1.5 Operator와 Delimiter

#### Operator (연산자)

| 분류   | 연산자                   | 예시              |
| ------ | ------------------------ | ----------------- |
| 산술   | +, -, \*, /, //, %, \*\* | 2 + 3, 5 \*\* 2   |
| 비교   | ==, !=, >, <, >=, <=     | x > 5, a == b     |
| 논리   | and, or, not             | x and y, not flag |
| 할당   | =, +=, -=, \*=, /=       | x = 10, x += 5    |
| 비트   | &, \|, ^, ~, <<, >>      | 5 & 3, 1 << 2     |
| 멤버십 | in, not in               | 'a' in 'abc'      |
| 동일성 | is, is not               | x is None         |

#### Delimiter (구분자)

| 구분자 | 용도            | 예시                  |
| ------ | --------------- | --------------------- |
| ()     | 함수 호출, 튜플 | print(), (1, 2)       |
| []     | 리스트, 인덱싱  | [1, 2, 3], arr[0]     |
| {}     | 딕셔너리, 집합  | {'a': 1}, {1, 2}      |
| :      | 코드 블록 시작  | if x:, def func():    |
| ,      | 요소 구분       | (1, 2, 3), func(a, b) |
| .      | 속성 접근       | obj.method()          |


In [12]:
# Operator와 Delimiter 예시
print("=== Operator 예시 ===")
x = 10  # = (할당)
y = x + 5  # + (산술)
is_positive = x > 0  # > (비교)
result = (x > 5) and (y < 20)  # and (논리)

print(f"x = {x}")
print(f"y = x + 5 = {y}")
print(f"x > 0 = {is_positive}")
print(f"(x > 5) and (y < 20) = {result}")
print()

print("=== Delimiter 예시 ===")
my_list = [1, 2, 3]  # [] 리스트
my_tuple = (4, 5, 6)  # () 튜플
my_dict = {'a': 1, 'b': 2}  # {} 딕셔너리, : 키-값 구분

print(f"list: {my_list}")
print(f"tuple: {my_tuple}")
print(f"dict: {my_dict}")
print(f"dict['a'] = {my_dict['a']}")

=== Operator 예시 ===
x = 10
y = x + 5 = 15
x > 0 = True
(x > 5) and (y < 20) = True

=== Delimiter 예시 ===
list: [1, 2, 3]
tuple: (4, 5, 6)
dict: {'a': 1, 'b': 2}
dict['a'] = 1


## 2. Literal 심화

### 2.1 Literal이란?

Literal은 코드에 값을 직접 적는 표기다. `x = 42`에서 `42`는 Literal이고, `x`는 그 값을 가리키는 이름이다.

```python
x = 42
name = "Alice"
pi = 3.14
```


### 2.2 Numeric Literals

#### Integer Literals

**표기법**:

- 10진수: 42, 1_000_000
- 2진수: 0b1010, 0b1111_0000
- 8진수: 0o12, 0o755
- 16진수: 0xA, 0xFF, 0x1A2B


In [13]:
# Integer Literals
print("=== Integer Literals ===")
print()

# 10진수
decimal = 42
large = 1_000_000  # 언더스코어로 가독성 향상
print(f"10진수: {decimal}, {large:,}")

# 2진수 (0b)
binary = 0b1010
print(f"2진수 0b1010 = {binary}")

# 8진수 (0o)
octal = 0o12
print(f"8진수 0o12 = {octal}")

# 16진수 (0x)
hexadecimal = 0xA
hex_ff = 0xFF
print(f"16진수 0xA = {hexadecimal}, 0xFF = {hex_ff}")
print()

# 진법 변환
number = 255
print(f"10진수 {number}를 다른 진법으로:")
print(f"  2진수: {bin(number)}")
print(f"  8진수: {oct(number)}")
print(f"  16진수: {hex(number)}")

=== Integer Literals ===

10진수: 42, 1,000,000
2진수 0b1010 = 10
8진수 0o12 = 10
16진수 0xA = 10, 0xFF = 255

10진수 255를 다른 진법으로:
  2진수: 0b11111111
  8진수: 0o377
  16진수: 0xff


#### Float Literals

**표기법**:

- 일반: 3.14, 0.5, 5.0
- 과학적: 1e10, 1.5e-3, 2.5E6


In [14]:
# Float Literals
import math
print("=== Float Literals ===")
print()

# 일반 표기법
pi = 3.14
half = 0.5
five = 5.0
print(f"일반 표기: {pi}, {half}, {five}")

# 과학적 표기법
large = 1e10  # 1 × 10^10
small = 1.5e-3  # 1.5 × 10^-3
print(f"1e10 = {large}")
print(f"1.5e-3 = {small}")
print()

# 부동소수점 오차 주의!
print("부동소수점 오차:")
result = 0.1 + 0.2
print(f"0.1 + 0.2 = {result}")
print(f"0.1 + 0.2 == 0.3? {result == 0.3}")
print()

# 해결법
print(f"round(0.1 + 0.2, 1) == 0.3? {round(result, 1) == 0.3}")
print(f"math.isclose(0.1 + 0.2, 0.3)? {math.isclose(result, 0.3)}")

=== Float Literals ===

일반 표기: 3.14, 0.5, 5.0
1e10 = 10000000000.0
1.5e-3 = 0.0015

부동소수점 오차:
0.1 + 0.2 = 0.30000000000000004
0.1 + 0.2 == 0.3? False

round(0.1 + 0.2, 1) == 0.3? True
math.isclose(0.1 + 0.2, 0.3)? True


#### Complex Literals

**표기법**:

- 1+2j, 3.14+0j, -5j
- complex(1, 2)


In [15]:
# Complex Literals
print("=== Complex Literals ===")
print()

c1 = 1+2j
c2 = complex(3, 4)
print(f"1+2j = {c1}")
print(f"complex(3, 4) = {c2}")
print()

# 복소수 연산
result = c1 + c2
print(f"{c1} + {c2} = {result}")
print()

# 복소수 속성
c = 3+4j
print(f"c = {c}")
print(f"  실수부: {c.real}")
print(f"  허수부: {c.imag}")
print(f"  켤레: {c.conjugate()}")
print(f"  절댓값: {abs(c)}")

=== Complex Literals ===

1+2j = (1+2j)
complex(3, 4) = (3+4j)

(1+2j) + (3+4j) = (4+6j)

c = (3+4j)
  실수부: 3.0
  허수부: 4.0
  켤레: (3-4j)
  절댓값: 5.0


### 2.3 String Literals

#### 기본 표기법

- 작은따옴표: 'Hello'
- 큰따옴표: "Hello"
- 삼중따옴표: '''Hello''', """Hello"""


In [ ]:
# String Literals 기본
print("=== String Literals ===")
print()

# 따옴표 종류
str1 = 'single'
str2 = "double"
str3 = '''triple
multi-line'''

print(f"작은따옴표: {str1}")
print(f"큰따옴표: {str2}")
print(f"삼중따옴표:\n{str3}")
print()

# 따옴표 혼용
str4 = "It's a beautiful day"
str5 = 'He said "Hello"'
print(f"혼용1: {str4}")
print(f"혼용2: {str5}")

=== String Literals ===

작은따옴표: single
큰따옴표: double
삼중따옴표:
triple
multi-line

혼용1: It's a beautiful day
혼용2: He said "Hello"


'\n다중 주석\n'

#### 특수 문자열 접두사

1. **r-string (Raw)**: 이스케이프 무시
2. **f-string (Formatted)**: 변수 삽입
3. **b-string (Bytes)**: 바이트 문자열


In [21]:
# 특수 문자열 접두사
print("=== 특수 문자열 접두사 ===")
print()

# r-string (Raw)
# normal = 'C:\\Users\\name'
normal = 'C:\\Users\\name'
raw = r'C:\\Users\name'
print(f"일반: {normal}")
print(f"Raw: {raw}")
print()

# f-string (Formatted)
name = "Alice"
age = 25
greeting = f"이름: {name}, 나이: {age}"
print(f"f-string: {greeting}")
print(f"계산: 10 + 5 = {10 + 5}")
print()

# b-string (Bytes)
byte_str = b'Hello'
print(f"Bytes: {byte_str}")
print(f"타입: {type(byte_str)}")
print(f"디코드: {byte_str.decode('utf-8')}")

=== 특수 문자열 접두사 ===

일반: C:\Users\name
Raw: C:\\Users\name

f-string: 이름: Alice, 나이: 25
계산: 10 + 5 = 15

Bytes: b'Hello'
타입: <class 'bytes'>
디코드: Hello


#### Escape Sequences

| 시퀀스 | 의미       |
| ------ | ---------- |
| \n     | 줄바꿈     |
| \t     | 탭         |
| \\     | 백슬래시   |
| \'     | 작은따옴표 |
| \"     | 큰따옴표   |


In [ ]:
# Escape Sequences
print("=== Escape Sequences ===")
print()

# 줄바꿈
print("첫 줄\n두 번째 줄\n세 번째 줄")
print()

# 탭
print("이름\t나이\t직업")
print("Alice\t25\t개발자")
print()

# 따옴표
print("He said \"Hello\"")
print('It\'s a beautiful day')

=== Escape Sequences ===

첫 줄
두 번째 줄
세 번째 줄

이름	나이	직업
Alice	25	개발자

He said "Hello"
It's a beautiful day


### 2.4 Boolean & None Literals

**Boolean**: True, False (첫 글자 대문자 필수)
**None**: 값이 없음을 나타냄


In [23]:
# Boolean & None
print("=== Boolean & None ===")
print()

# Boolean
is_active = True
is_valid = False
print(f"is_active: {is_active}")
print(f"is_valid: {is_valid}")
print()

# Truthy/Falsy
print("Truthy/Falsy:")
print(f"  bool(0) = {bool(0)}")
print(f"  bool(1) = {bool(1)}")
print(f"  bool('') = {bool('')}")
print(f"  bool('text') = {bool('text')}")
print()

# None
result = None
print(f"result = {result}")
print(f"result is None? {result is None}")
print(f"bool(None) = {bool(None)}")

=== Boolean & None ===

is_active: True
is_valid: False

Truthy/Falsy:
  bool(0) = False
  bool(1) = True
  bool('') = False
  bool('text') = True

result = None
result is None? True
bool(None) = False


### 2.5 Container Literals


In [24]:
# Container Literals
print("=== Container Literals ===")
print()

# List
my_list = [1, 2, 3]
print(f"List: {my_list}")

# Tuple
my_tuple = (1, 2, 3)
single = (42,)  # 1개 요소는 쉼표 필수!
print(f"Tuple: {my_tuple}")
print(f"1개 Tuple: {single}")

# Set
my_set = {1, 2, 3}
print(f"Set: {my_set}")

# Dict
my_dict = {'a': 1, 'b': 2}
print(f"Dict: {my_dict}")
print()

# 빈 컨테이너 주의!
empty_list = []
empty_tuple = ()
empty_set = set()  # {}는 딕셔너리!
empty_dict = {}

print("빈 컨테이너:")
print(f"  list: {empty_list}")
print(f"  tuple: {empty_tuple}")
print(f"  set: {empty_set}")
print(f"  dict: {empty_dict}")

=== Container Literals ===

List: [1, 2, 3]
Tuple: (1, 2, 3)
1개 Tuple: (42,)
Set: {1, 2, 3}
Dict: {'a': 1, 'b': 2}

빈 컨테이너:
  list: []
  tuple: ()
  set: set()
  dict: {}


## 3. Code Tokenization

### 3.1 tokenize 모듈 실전


In [25]:
# tokenize 모듈 실습
import tokenize
import io

code = "x = 42"
tokens = tokenize.generate_tokens(io.StringIO(code).readline)

print("=== Token 상세 분석 ===")
print(f"코드: {code}\n")

for token in tokens:
  print(f"Token:")
  print(f"  type: {tokenize.tok_name[token.type]}")
  print(f"  string: '{token.string}'")
  print(f"  start: {token.start}")
  print(f"  end: {token.end}")
  print()

=== Token 상세 분석 ===
코드: x = 42

Token:
  type: NAME
  string: 'x'
  start: (1, 0)
  end: (1, 1)

Token:
  type: OP
  string: '='
  start: (1, 2)
  end: (1, 3)

Token:
  type: NUMBER
  string: '42'
  start: (1, 4)
  end: (1, 6)

Token:
  type: NEWLINE
  string: ''
  start: (1, 6)
  end: (1, 7)

Token:
  type: ENDMARKER
  string: ''
  start: (2, 0)
  end: (2, 0)



In [26]:
# 복잡한 코드 분석
complex_code = """
def calculate(price, quantity):
    total = price * quantity
    return total
"""

print("=== 복잡한 코드 Token 분석 ===")
print("코드:")
print(complex_code)
print("\nToken 목록:")

tokens = tokenize.generate_tokens(io.StringIO(complex_code).readline)

for token in tokens:
  if token.type not in [tokenize.ENCODING, tokenize.ENDMARKER, tokenize.NL]:
    token_type = tokenize.tok_name[token.type]
    print(f"{token_type:12} : '{token.string}'")

=== 복잡한 코드 Token 분석 ===
코드:

def calculate(price, quantity):
    total = price * quantity
    return total


Token 목록:
NAME         : 'def'
NAME         : 'calculate'
OP           : '('
NAME         : 'price'
OP           : ','
NAME         : 'quantity'
OP           : ')'
OP           : ':'
NEWLINE      : '
'
INDENT       : '    '
NAME         : 'total'
OP           : '='
NAME         : 'price'
OP           : '*'
NAME         : 'quantity'
NEWLINE      : '
'
NAME         : 'return'
NAME         : 'total'
NEWLINE      : '
'
DEDENT       : ''


### 3.2 PEP 8 스타일 가이드

#### 주요 규칙

1. **들여쓰기**: 공백 4개
2. **한 줄 길이**: 최대 79자
3. **빈 줄**: 함수/클래스 사이 2줄
4. **공백**: 연산자 양쪽, 쉼표 뒤


In [ ]:
# PEP 8 예시
print("=== PEP 8 스타일 ===")
print()

# PEP 8에 맞게 정리한 코드


def calculate_total(price, quantity):
  """총액 계산"""
  result = price * quantity
  return result


class UserAccount:
  """사용자 계정"""

  def __init__(self, name, age):
    self.name = name
    self.age = age


# 상수
MAX_SIZE = 1000
PI = 3.14159

print("PEP 8 준수 코드 작성 완료")

### 3.3 코드 스타일 도구

다음 도구는 설치 후 터미널에서 실행한다. pylint는 코드 품질을, flake8은 스타일 위반 등을 검사한다. black은 코드 형식을 자동으로 정리한다.

```bash
pylint my_script.py
flake8 my_script.py
black my_script.py
```


## 실습 과제

### 과제 1: Keyword 분류 (10점)


In [ ]:
# 과제 1: Keyword 분류
import keyword

# TODO: 모든 Keyword를 분류별로 정리하여 출력
# TODO: Hard/Soft Keyword 구분

pass

### 과제 2: Literal 종합 실습 (15점)


In [ ]:
# 과제 2: 모든 Literal 작성

# TODO 1: Integer (10진, 2진, 8진, 16진)
integers = [10, 0b1010, 0o12, 0xA, 1_000_000]

# TODO 2: Float (일반, 과학적 표기)
floats = []

# TODO 3: String (일반, raw, f-string)
strings = []

# TODO 4: Container (list, tuple, set, dict)
containers = []

pass

### 과제 3: Token 분석기 (20점)


In [ ]:
# 과제 3: Token 분석 프로그램
import tokenize
import io
from collections import Counter


def analyze_code(code):
  """코드를 분석하여 Token 통계 출력"""
  # TODO: Token 타입별 개수
  # TODO: Keyword 목록
  # TODO: Identifier 목록
  pass


test_code = """
def add(a, b):
    return a + b

result = add(10, 5)
"""

analyze_code(test_code)

### 과제 4: PEP 8 검사기 (15점)


In [ ]:
# 과제 4: 간단한 PEP 8 검사기

def check_pep8(code):
  """PEP 8 위반 사항 검사"""
  issues = []

  # TODO: 줄 길이 체크 (79자)
  # TODO: 들여쓰기 체크 (4의 배수)
  # TODO: 명명 규칙 체크

  return issues


bad_code = """
def CalculateTotal(Price,Quantity):
   Total=Price*Quantity
   return Total
"""

check_pep8(bad_code)

## 토론 주제

1. Keyword를 새로 추가하면 기존 코드에 어떤 영향이 있을까?
2. Soft Keyword는 기존 식별자와 충돌하는 문제를 어떻게 줄일까?
3. f-string 안의 Literal 부분과 Expression 부분은 어떻게 구분할 수 있을까?


## 복습

1. `tokenize`의 출력에서 Keyword와 Identifier를 어떻게 구분할 수 있을까?
2. 같은 정수를 10진수, 2진수, 8진수, 16진수로 적어보자.
3. 일반 문자열, raw string, f-string이 각각 필요한 예를 들어보자.
4. 식별자의 문법 규칙과 PEP 8의 명명 관례는 무엇이 다른가?


## 참고 자료

### 필수

- [Keywords](https://dsaint31.tistory.com/510)
- [Literal](https://dsaint31.tistory.com/462)
- [Type Summary](https://dsaint31.tistory.com/515)

### 권장

- [PEP 8](https://www.python.org/dev/peps/pep-0008/)
- [Naming Convention](http://ds31x.blogspot.com/2023/07/etc-naming-convention.html)

**다음 수업: Expression & Statement**
